# 实验一 · Hello World 与 Fork-Join 模型

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐ 入门　|　**预计时长**：10–20 分钟

本实验为第五章的入门案例，不涉及性能优化。其目标是阐明 OpenMP 最基本的执行模型：一条编译制导语句如何在源代码中划出一段并行区域，程序又如何在该区域的入口派生（fork）出线程组、在出口汇合（join）。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 本实验没有性能表格，其全部内容都围绕**执行模型**展开。后续七个实验的性能讨论，都建立在本实验所建立的模型之上。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明 OpenMP 与第四章 Pthread 在**抽象层次**上的区别，理解「编译制导」这一形式的含义
- 掌握 `#pragma omp parallel` 的语法结构，识别指令、子句与结构化块三个组成部分
- 理解 **Fork-Join 模型**：并行区域入口的线程派生、出口的隐式栅栏与汇合
- 正确区分 `omp_get_thread_num()` 与 `omp_get_num_threads()`，并解释后者在串行区域中为何恒为 1
- 掌握设定线程数的三种途径（`num_threads` 子句、`omp_set_num_threads()`、`OMP_NUM_THREADS` 环境变量）及其优先级
- 理解 `_OPENMP` 宏的作用，能够写出在无 OpenMP 支持时仍可编译的条件代码

## 🗺️ 学习路径

1. **准备阶段**：回顾第四章 Pthread 的线程创建方式，建立「显式管理」与「编译制导」的对照
2. **V0 · 串行区域**：在并行区域之外调用运行时库函数，观察其返回值
3. **V1 · 并行区域**：以 `#pragma omp parallel` 创建线程组，观察输出次序的不确定性
4. **线程数设定对照**：以三种方式指定线程数量，验证其优先级关系
5. **分析与延伸**：讨论输出乱序的成因，以及隐式栅栏的位置

## 1. 背景知识：从显式线程管理到编译制导

### 1.1 第四章的实现方式

在第四章中，创建一组线程需要完成以下步骤：声明 `pthread_t` 数组、为每个线程准备参数结构体、循环调用 `pthread_create` 并传入函数指针、再循环调用 `pthread_join` 等待全部线程结束。线程函数本身还必须写成 `void *f(void *arg)` 的固定形式，参数的打包与拆包全部由程序员负责。

该方式具有较强的表达能力，但存在以下两方面局限：

- **线程管理代码的规模显著超过核心计算逻辑**。一段十余行的计算逻辑，其外围的线程管理代码常达数十行。
- **串行版本与并行版本无法共存于同一份源文件**。一旦改为多线程结构，串行代码的原有结构即被破坏，调试时难以回退到串行状态进行对照。

### 1.2 OpenMP 的实现方式

OpenMP 采用**编译制导**（compiler directive）的形式：程序员保留原有的串行代码，只在需要并行的位置插入以 `#pragma omp` 起始的编译制导语句，由编译器负责生成线程创建、参数传递与同步的全部代码。

这一形式带来一个重要性质：**不支持 OpenMP 的编译器将忽略这些编译制导语句，源程序的串行语义不受影响**（即制导语句对该类编译器是透明的）。C 语言标准规定，编译器遇到无法识别的 `#pragma` 应当忽略它。因此同一份源代码，加上 `-fopenmp` 编译得到并行程序，去掉该选项则编译得到语义正确的串行程序。这一性质使得「先保证正确性，再实施性能优化」的开发次序成为可能。

> **需要留意的前提**：上述性质成立的条件是程序中不出现 OpenMP 运行时库函数（如 `omp_get_thread_num()`）。一旦调用了这些函数，去掉 `-fopenmp` 便会产生链接错误。第 3 节将说明如何用 `_OPENMP` 宏处理这一情况。

### 1.3 两者的关系

OpenMP 并非 Pthread 的替代品，而是建立在其之上的更高层抽象。在 Linux 平台上，GCC 的 OpenMP 运行时库（libgomp）即基于 Pthread 实现。第四章所讨论的数据竞争、临界区、栅栏等概念在本章依然全部有效，只是表达它们的语法形式发生了变化。

| 关注点 | 第四章 Pthread | 第五章 OpenMP |
|---|---|---|
| 线程创建 | `pthread_create` / `pthread_join` | `#pragma omp parallel` |
| 任务划分 | 由程序员显式计算每个线程的区间 | `#pragma omp for` 自动划分 |
| 互斥 | `pthread_mutex_t` | `#pragma omp critical` / `atomic` |
| 栅栏 | `pthread_barrier_t` | `#pragma omp barrier`（多数场合为隐式） |
| 归约 | 显式加锁累加 | `reduction` 子句 |
| 串并共存 | 不支持 | 支持（去掉 `-fopenmp` 即编译为串行程序） |

## 2. 执行模型：Fork-Join

OpenMP 程序的执行遵循 **Fork-Join 模型**，其过程可分为四个阶段：

```text
                    ┌── 线程 1 ──┐
                    │            │
  主线程 ───────────┼── 线程 2 ───┼────────── 主线程 ────►
        (串行区)    │            │  (隐式栅栏)  (串行区)
                    └── 线程 3 ──┘
                    ▲             ▲
                   Fork          Join
```

1. **串行执行**。程序自 `main` 起以单线程运行，此线程称为**主线程**。

   > **术语说明**：主线程在 OpenMP 4.5 及更早的规范中记作 master thread；自 OpenMP 5.0 起，规范已将其改称 **primary thread**，`master` 构造亦由 `masked` 构造取代，master 一词仅作为遗留术语保留。本教材统一使用中文译名「主线程」，阅读英文文档时需注意两种写法指向同一概念。

2. **Fork**。主线程遇到 `#pragma omp parallel` 时创建若干个新线程，与主线程共同构成**线程组**（team）。组内每个线程都获得一个从 0 开始的编号，主线程的编号恒为 0。需要指出的是，OpenMP 规范中并不存在「从线程」这一概念：组内各线程地位对等，主线程与其余线程执行的是完全相同的结构化块，差别仅在于线程编号，以及主线程在并行区域结束后继续存在。
3. **并行执行**。**线程组中的每一个线程都完整执行一遍结构化块中的全部语句**。此处是初学阶段的常见误解所在：`parallel` 构造本身并**不**划分工作，它只是使线程组中的各个线程同时执行同一个结构化块。工作的划分需借助 `#pragma omp for` 等工作分担构造（worksharing construct）。
4. **Join**。结构化块的右花括号处存在一个**隐式栅栏**：先到达的线程必须等待其余线程，全部到齐后，组内其余线程终止，主线程继续向下执行。

### 2.1 右花括号的三重含义

并行区域结尾的 `}` 同时对应以下三方面语义，阅读代码时均需一并考虑：

| 语义 | 说明 |
|---|---|
| 隐式栅栏 | 组内全部线程均在此同步 |
| 线程组解散 | 组内其余线程在此终止，线程组不再存在 |
| 数据环境终结 | 区域内声明的私有变量在此失效 |

由于隐式栅栏的存在，并行区域的耗时由**执行时间最长的线程**决定。

### 2.2 输出次序为何不确定

线程组内各线程之间不存在任何预定的执行次序，其调度由操作系统负责。各线程被调度到处理器的时刻，以及完成 `printf` 调用的先后，均取决于运行时的调度结果。因此本实验的输出顺序在每次运行时都可能不同。该现象属于并行执行的**固有特性**，而非程序缺陷。

反之，若某段并行代码的正确性依赖于线程输出的先后顺序，则该代码本身存在逻辑缺陷。

## 3. OpenMP 关键知识点

### 3.1 编译制导语句的结构

```c
#pragma omp parallel num_threads(4)
└──┬──┘ └┬┘ └───┬──┘ └──────┬─────┘
   │     │      │           └── 子句（clause），可有零到多个
   │     │      └────────────── 指令（directive），指明构造的类型
   │     └───────────────────── 标识此为 OpenMP 制导语句
   └─────────────────────────── C 预处理指令
```

制导语句之后必须紧跟一个**结构化块**（structured block）：单条语句，或一对花括号包裹的复合语句。结构化块只允许有一个入口和一个出口，块内不得出现 `goto` 跳出、`return`、`break` 越界等改变控制流的语句。

### 3.2 三个基础运行时库函数

使用这些函数需要包含头文件 `<omp.h>`。

| 函数 | 返回值 | 在串行区域中的取值 |
|---|---|---|
| `omp_get_thread_num()` | 调用线程在当前线程组中的编号 | `0`（主线程） |
| `omp_get_num_threads()` | **当前线程组**的线程数量 | `1` |
| `omp_get_num_procs()` | 运行时可见的处理器数量 | 与位置无关 |

> **常见误解**：在并行区域**之外**调用 `omp_get_num_threads()`，并期望由此获得后续并行区域将要使用的线程数。该函数返回的是调用时刻**当前**线程组的规模，而串行区域中的线程组只有主线程一个成员，因此返回值恒为 1。若要查询后续并行区域的默认线程数，应使用 `omp_get_max_threads()`。

### 3.3 设定线程数的三种途径

按优先级由高到低排列：

| 途径 | 写法 | 作用范围 |
|---|---|---|
| 1. `num_threads` 子句 | `#pragma omp parallel num_threads(4)` | 仅本次并行区域 |
| 2. 运行时库函数 | `omp_set_num_threads(4);` | 此后所有未带子句的并行区域 |
| 3. 环境变量 | `export OMP_NUM_THREADS=4` | 整个进程 |

三者同时存在时，高优先级者覆盖低优先级者。本章的实验代码统一采用第一种方式，把线程数作为命令行参数传入，便于在同一次实验中比较不同线程数下的运行结果。

### 3.4 `_OPENMP` 宏与条件编译

编译器在开启 `-fopenmp` 时会自动定义宏 `_OPENMP`，其值是一个十进制整数常量，格式为 **`yyyymm`**：前四位为该实现所遵循的 OpenMP 规范的发布年份，后两位为发布月份（月份不足两位时补前导零）。例如 `201511` 表示 2015 年 11 月发布的 OpenMP 4.5 规范。由于该编码随时间单调递增，可直接用 `>=` 进行版本比较。

| `_OPENMP` 取值 | 对应规范 |
|---|---|
| 200805 | OpenMP 3.0 |
| 201107 | OpenMP 3.1 |
| 201307 | OpenMP 4.0 |
| 201511 | OpenMP 4.5 |
| 201811 | OpenMP 5.0 |

该宏有两种典型用法。其一是**强制检查**，用于确保编译选项正确，本章全部实验源码均采用这一写法：

```c
#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif
```

其二是**能力检测**，用于在高版本规范提供的特性不可用时回退至兼容实现。实验六将用它判断编译器是否支持 OpenMP 4.5 引入的数组段归约：

```c
#if _OPENMP >= 201511
  // 使用 reduction(+ : bins[:num_bins])
#else
  // 退回到手工分配私有数组再合并
#endif
```

## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_hello'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

## 6. 版本设计总览

本实验的源文件 `omp_hello.c` 包含两段互相对照的代码：

| 版本 | 位置 | 考察点 | 预期观察 |
|---|---|---|---|
| **V0** | 并行区域之前 | 串行区域中的线程组规模 | `omp_get_num_threads()` 返回 1 |
| **V1** | `#pragma omp parallel` 之内 | 线程组的创建与并行执行 | 输出 N 行，顺序不定 |

两段代码调用的是**同一组运行时库函数**，唯一的差别在于它们所处的位置。对这一差别单独加以对照，即可说明这些函数的返回值取决于**调用位置**，而与程序其他位置是否出现过并行区域无关。

## 7. 逐版本代码讲解

### 7.1 V0 · 串行区域

```c
printf("  omp_get_thread_num()  = %d\n", omp_get_thread_num());
printf("  omp_get_num_threads() = %d\n", omp_get_num_threads());
```

此处尚未进入任何并行区域，程序仍以单线程运行。当前线程组仅包含主线程一个成员，因此两个函数分别返回 `0` 与 `1`。

需注意：即便程序稍后会创建 4 个线程，此处的 `omp_get_num_threads()` **依然返回 1**。运行时库函数报告的是调用发生时刻的状态，其返回值不反映后续并行区域的线程组规模。

### 7.2 V1 · 并行区域

```c
#pragma omp parallel num_threads(thread_count)
{ hello(); }
```

该制导语句经编译器处理后，其效果相当于第四章中 `pthread_create` 与 `pthread_join` 两个循环的组合。`num_threads(thread_count)` 子句指定本次创建的线程数量。

> **关于编译器的处理方式**：此处编译器所做的并非宏展开（宏展开是预处理阶段的文本替换）。GCC 的实际处理方式是**函数外提**（outlining）：把结构化块提取为一个独立的内部函数，再在原位置插入对运行时库 libgomp 的调用（如 `GOMP_parallel`），由该库负责线程的创建、参数传递与回收。

被调用的 `hello()` 函数定义如下：

```c
static void hello(void) {
  int my_rank = omp_get_thread_num();      // 各线程取值不同：0, 1, 2, ...
  int thread_count = omp_get_num_threads();  // 各线程取值相同：线程组规模
  printf("  Hello from thread %d of %d\n", my_rank, thread_count);
}
```

两个局部变量 `my_rank` 与 `thread_count` 声明在函数体内，位于每个线程各自的栈上，因而具有线程**私有**属性。这一点在第 3 节尚未展开，实验二将系统讨论数据环境的默认规则。

> **关于 `static` 修饰符**：本章全部源码对内部辅助函数一律加 `static`。这样做既限定了链接范围，也便于编译器实施内联，使各版本之间的性能对照不受函数调用开销的干扰。

### 7.3 隐式栅栏的验证

并行区域之后，程序再次打印 `omp_get_num_threads()`。此时线程组已经解散，返回值重新变为 1。这一处输出与 V0 相互印证，共同说明了并行区域的作用范围。

## 8. 源代码写入

下面的单元格将完整源码写入 `{SRC_DIR}/omp_hello.c`。该文件与课程代码目录中的版本逐字节一致，可直接在目标开发板上编译运行。

In [ ]:
%%writefile {SRC_DIR}/omp_hello.c
#define _POSIX_C_SOURCE 200809L

#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#define MAX_THREADS 16
#define BANNER "============================================================"

// The task executed independently by every thread of the team.
static void hello(void) {
  // Rank of the calling thread inside the current team, starting from 0.
  int my_rank = omp_get_thread_num();
  // Total number of threads in the current team.
  int thread_count = omp_get_num_threads();

  printf("  Hello from thread %d of %d\n", my_rank, thread_count);
}

int main(int argc, char *argv[]) {
  if (argc != 2) {
    printf("Usage: %s <thread_count>\n", argv[0]);
    printf("       1 <= thread_count <= %d\n", MAX_THREADS);
    return 1;
  }

  int thread_count = (int)strtol(argv[1], NULL, 10);
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }

  printf("%s\n", BANNER);
  printf(" Lab 1: Hello World and the Fork-Join Model\n");
  printf(" _OPENMP: %d | Procs: %d | Max threads: %d | Requested: %d\n",
         _OPENMP, omp_get_num_procs(), omp_get_max_threads(), thread_count);
  printf("%s\n", BANNER);

  // V0: the master thread runs alone, so the team size reported here is 1.
  printf("\n[V0] Serial region\n");
  printf("  omp_get_thread_num()  = %d\n", omp_get_thread_num());
  printf("  omp_get_num_threads() = %d\n", omp_get_num_threads());

  // V1: Fork. The directive creates a team of thread_count threads and every
  // thread executes the same structured block.
  printf("\n[V1] Parallel region\n");
#pragma omp parallel num_threads(thread_count)
  { hello(); }
  // The closing brace carries three meanings at once: an implicit barrier,
  // the join of the team and the end of the data environment of the region.

  printf("\n[V1] Back to the serial region\n");
  printf("  omp_get_num_threads() = %d\n", omp_get_num_threads());
  printf("  Hello from the master thread\n");

  return 0;
}

## 9. 编译与运行

### 9.1 编译

注意观察编译命令中的 `-fopenmp`。若去掉该选项，源码中的 `#error` 将立即中止编译——这正是 3.4 节所述强制检查的效果。

In [ ]:
bin_hello = compile_c('omp_hello.c', extra=())

### 9.2 以 4 线程运行

请重点观察三处输出：

1. `[V0]` 段中 `omp_get_num_threads()` 的返回值；
2. `[V1]` 段中四行线程输出的**先后顺序**；
3. 并行区域结束后 `omp_get_num_threads()` 的返回值。

In [ ]:
out4 = run_c(bin_hello, 4)

### 9.3 重复运行以观察输出次序的不确定性

同一个程序连续运行五次，比较五次输出的行序是否相同。若出现顺序不一致，即说明线程之间不存在任何预定的执行次序。

> **反之不成立**：若五次运行的次序恰好相同，并不能说明线程之间存在确定的执行次序。线程数较少、并行区域内工作量极小时，线程创建的先后往往主导了输出顺序，重复观察因而可能长期得到同一结果。非确定性意味着**任何一种次序都被允许出现**，重复实验只能证伪「存在固定次序」这一判断，而无法证实它。

In [ ]:
for trial in range(5):
    out = subprocess.run([bin_hello, '4'], capture_output=True,
                         text=True).stdout
    order = [line.split()[3] for line in out.splitlines()
             if line.strip().startswith('Hello from thread')]
    print('第 %d 次运行，线程输出次序：%s' % (trial + 1, ' -> '.join(order)))

## 10. 线程数设定方式的对照实验

本节验证 3.3 节所述的优先级关系。实验方法是：设置环境变量 `OMP_NUM_THREADS=2`，同时通过命令行参数要求程序使用 `num_threads(6)`，观察最终生效的是哪一个。

由于源码中的 `#pragma omp parallel` 带有 `num_threads` 子句，而该子句的优先级最高，预期结果是**创建 6 个线程**，环境变量被覆盖。

> **对照实验的读法**：本实验源码中的并行区域**始终**带有 `num_threads` 子句，因此下面两种情形中真正决定线程数的都是该子句。情形一中子句与环境变量取值一致，无法据此区分二者谁在起作用，它只作为**基线**；只有情形二中两者取值冲突，实际线程数才构成有效的优先级判据。环境变量本身是否生效，应通过程序头部 `Max threads` 一栏（即 `omp_get_max_threads()` 的返回值）来观察——该函数不受 `num_threads` 子句影响，详见 10.1 节。

In [ ]:
print('=' * 60)
print(' 情形一（基线）：环境变量 OMP_NUM_THREADS=2，num_threads 子句同为 2，取值一致')
print('=' * 60)
_ = run_c(bin_hello, 2, env={'OMP_NUM_THREADS': 2})

print()
print('=' * 60)
print(' 情形二（判据）：环境变量 OMP_NUM_THREADS=2，num_threads 子句请求 6，取值冲突')
print('=' * 60)
_ = run_c(bin_hello, 6, env={'OMP_NUM_THREADS': 2})

### 10.1 关于 `Max threads` 一栏

程序头部打印的 `Max threads` 来自 `omp_get_max_threads()`。该函数返回的是**未指定 `num_threads` 子句时**并行区域将会使用的线程数量，因此它会随 `OMP_NUM_THREADS` 变化，而不受 `num_threads` 子句影响。把它与 `Requested` 一栏对照阅读，可以清楚看出两者的分工。

## 11. 结果分析

**① 运行时库函数报告的是调用时刻的状态**

`omp_get_num_threads()` 在串行区域返回 1，在并行区域返回线程组规模，并行区域结束后再次返回 1。三次调用写的是同一行代码，返回值却各不相同，原因仅在于调用位置所处的区域不同。

**② `parallel` 构造使各线程执行完整的结构化块，而不划分工作**

四个线程各自完整执行了一遍 `hello()`，因此得到四行输出，而不是把工作量分成四份。若将一个循环直接置于 `parallel` 结构化块内，得到的结果将是**每个线程都完整执行一遍该循环**，总工作量变为原来的 N 倍（N 为线程数）。避免这一错误的方法是使用 `#pragma omp for`，实验二将正式引入该构造。

**③ 输出次序的不确定性是并行执行的固有特性**

五次重复运行往往给出不同的行序；即使某次实验中五次行序恰好相同，也不能推断线程之间存在确定的执行次序（参见 9.3 节的说明）。需要强调的是，次序的不确定性本身不构成缺陷；**依赖特定执行次序的程序**才是错误的。判断一段并行代码是否正确，标准之一就是：任意调度次序下结果是否一致。

**④ 子句的优先级高于环境变量**

第 10 节的对照实验表明，`num_threads(6)` 覆盖了 `OMP_NUM_THREADS=2`。其设计意图在于：环境变量提供全局默认值，便于部署时统一调整；子句提供局部精确控制，以满足程序自身的需要。

## 12. 🔧 动手练习

**练习 1**　修改线程数量，依次以 1、2、8、16 个线程运行程序，观察当线程数超过物理核心数时输出是否仍然正确。

**练习 2**　把 `#pragma omp parallel` 一行注释掉后重新编译运行，记录输出的变化，并解释为何程序**仍然可以正确编译**。

**练习 3**　去掉编译命令中的 `-fopenmp` 重新编译，记录错误信息，并说明是源码中的哪一处结构产生了该错误。

**练习 4**（进阶）　在 `hello()` 函数中新增一个在函数体外声明的全局计数器，让每个线程对其执行 `counter++`，运行多次观察最终结果是否恒为线程数。该现象将在实验二与实验六中系统讨论。

In [ ]:
# 练习 1 的脚手架：不同线程数下的输出
for nt in (1, 2, 8, 16):
    print('=' * 60)
    print(' 线程数 =', nt)
    print('=' * 60)
    out = subprocess.run([bin_hello, str(nt)], capture_output=True,
                         text=True).stdout
    lines = [l for l in out.splitlines()
             if l.strip().startswith('Hello from thread')]
    print('实际输出行数：%d' % len(lines))
    for l in lines:
        print(l)
    print()

## 13. 🤔 思考题

**思考题 1**　并行区域内的 `printf` 输出为何不会出现字符层面的交错（例如两行内容相互穿插）？这一保证来自 OpenMP 还是来自 C 标准库？如果把 `printf` 换成两次 `fputs` 调用，结论是否依然成立？

**思考题 2**　若把 `hello()` 中的 `int my_rank` 改为在函数外部声明的全局变量，程序的输出会发生什么变化？请从「变量存放在何处」的角度作出解释。

**思考题 3**　第四章中，创建线程需要显式调用 `pthread_join` 等待其结束；本实验的代码中并没有出现任何等待语句，程序却能保证四行输出全部完成后才继续。这一保证是由什么机制提供的？

**思考题 4**　假设某台机器有 4 个物理核心，程序请求创建 16 个线程。此时 `omp_get_num_threads()` 返回多少？程序的执行时间会是 4 线程时的四分之一、四倍，抑或是其他关系？请说明理由。

**思考题 5**　OpenMP 制导语句可被不支持它的编译器忽略，从而使同一份源码既能编译为串行程序也能编译为并行程序。本实验的源码是否具备这一性质？若不具备，是哪一部分破坏了它？应如何改写才能恢复？

## 14. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| 编译制导 | `#pragma omp` + 指令 + 子句 + 结构化块 |
| Fork-Join | 入口派生线程组，出口经隐式栅栏后汇合 |
| `parallel` 构造 | 线程组中每个线程各自**完整执行**结构化块，不进行工作划分 |
| 线程编号 | `omp_get_thread_num()`，主线程恒为 0 |
| 线程组规模 | `omp_get_num_threads()`，串行区域中恒为 1 |
| 线程数设定 | 子句 > 库函数 > 环境变量 |
| `_OPENMP` 宏 | 取值形如 201511，可用于强制检查与能力检测 |

### 与后续实验的衔接

本实验建立了执行模型，但尚未涉及两个关键问题：

- **工作如何划分**？`parallel` 只是使各线程执行同一结构化块，要让各线程分担不同的迭代，需要工作分担构造 `#pragma omp for`。→ 实验二、实验三
- **变量的共享与私有属性如何确定**？本实验中所有变量都是函数内的局部变量，其私有性由栈的存储特性保证。一旦并行区域需要访问外层变量，就必须明确其共享或私有属性。→ 实验二

下一个实验以梯形积分法为载体，同时回答这两个问题。